# Adaptive AM-FM Decomposition of Speech for Parkinson's Disease Classification
## SVM classification on the PC-GITA vowels

Classifies speakers with Parkinson's disease (PD) versus healthy controls (HC) from sustained vowels, using eaQHM harmonic AM/FM variation features ($H_1$–$H_5$) together with the normalised first differences of $f_0$ and $A_1$ and spectral / Teager-energy descriptors (18 features).

An RBF-kernel SVM is evaluated with repeated nested speaker-independent cross-validation on predefined folds (`master_cv_folds_vowels.csv`), stratified by label and gender. Hyper-parameters ($C$, $\gamma$) are tuned in a 5-fold inner loop by ROC AUC, and results are reported at sample and speaker level.

### Imports

In [ ]:
import pandas as pd
import numpy as np
from scipy.io import loadmat
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             confusion_matrix, precision_score, recall_score,
                             brier_score_loss)
from typing import Tuple
import warnings
warnings.filterwarnings("ignore")

### 0. Data preparation

In [ ]:
# ==========================================
# 0. DATA PREP
# ==========================================
data_aeiou = loadmat(f'../features/pc_gita_vowels_16k_5ms_chopped_650.mat')
data_aeiou = data_aeiou['results'].squeeze()
data_aeiou = pd.DataFrame(data_aeiou)
data_aeiou.columns = ['centroid_mean', 'centroid_std', 'spectral_flux_mean', 'spectral_flux_max',
                       'tpe_mean', 'tpe_std', 'am_fm_corr', 'ampl_var', 'freq_var', 'f0_var',
                       'SRER', 'f0_norm_diff', 'jitter_T', 'A1_norm_diff', 'spectral_slope',
                       'f0_entropy', 'name']

file_names = data_aeiou['name'].apply(lambda x: x[0])
data = data_aeiou.copy()

for col in ['centroid_mean', 'centroid_std', 'spectral_flux_mean', 'spectral_flux_max',
            'tpe_mean', 'tpe_std', 'am_fm_corr', 'f0_var', 'SRER',
            'f0_norm_diff', 'jitter_T', 'A1_norm_diff', 'spectral_slope', 'f0_entropy']:
    data[col] = data[col].apply(lambda x: x[0][0])


data['speaker'] = file_names.str.extract(r'(AVPEPUDEA[C]?\d{4})')[0]

gender_data = pd.read_csv("../gender_metadata/genders_pc_gita.csv", sep=",", header=0)
data = data.merge(gender_data, how="left", on="speaker")
data['label']  = data['speaker'].str.contains('C').map({True: 0, False: 1})
data['gender'] = data['gender'].map({'female': 0, 'male': 1})

ampl_expanded = data['ampl_var'].apply(lambda x: x.flatten())
freq_expanded = data['freq_var'].apply(lambda x: x.flatten())
for i in range(5):
    data[f'ampl_var_H{i+1}'] = ampl_expanded.apply(lambda v: v[i])
    data[f'freq_var_H{i+1}'] = freq_expanded.apply(lambda v: v[i])

data = data.drop(columns=['ampl_var', 'freq_var'])
data['vowel'] = file_names.str.extract(r'AVPEPUDEA[C]?\d{4}([aeiouAEIOU])')[0].str.upper()

print(data.head())

### 1. Load folds and merge into data

In [ ]:
# ==========================================
# 1. LOAD FOLDS AND MERGE INTO DATA
# ==========================================
fold_file = f"../folds/master_cv_folds_vowels.csv"
print(f"Loading predefined folds from {fold_file}...")
fold_map = pd.read_csv(fold_file)
fold_map['speaker'] = fold_map['speaker'].astype(str).str.strip().str.upper()

repeat_fold_cols = [c for c in fold_map.columns if c.startswith("Repeat_") and c.endswith("_Fold")]
speaker_folds = fold_map[['speaker'] + repeat_fold_cols].drop_duplicates(subset='speaker')

data['speaker'] = data['speaker'].astype(str).str.strip().str.upper()
data = data.merge(speaker_folds, on='speaker', how='inner').reset_index(drop=True)

if data.empty:
    raise ValueError("CRITICAL: DataFrame empty after fold merge — check speaker ID format")

for col in repeat_fold_cols:
    assert data.groupby('speaker')[col].nunique().max() == 1, \
        f"Speaker has inconsistent fold assignments in {col}"
print("✅ All speakers have consistent fold assignments")

original_speakers = set(fold_map['speaker'].str.strip().str.upper())
new_speakers = set(data['speaker'].str.strip().str.upper())
missing = new_speakers - original_speakers
if missing:
    print(f"⚠️  {len(missing)} speakers not in fold file: {missing}")

### 2. Feature set (all features)

In [ ]:
# ==========================================
# 2. DEFINE FEATURE SET (ALL FEATURES)
# ==========================================
AM_FEATURES   = [f'ampl_var_H{i+1}' for i in range(5)] + ['A1_norm_diff']
FM_FEATURES   = [f'freq_var_H{i+1}' for i in range(5)] + ['f0_norm_diff']
REST_FEATURES = ['centroid_mean', 'centroid_std', 'spectral_flux_mean', 'spectral_flux_max', 'tpe_mean',
                 'tpe_std']

ALL_FEATURES = AM_FEATURES + FM_FEATURES + REST_FEATURES

### 3. Groups and shared settings

In [ ]:
# ==========================================
# 3. FINALIZE GROUPS / SHARED META
# ==========================================
groups = data['speaker'].values
data['stratify_key'] = data['label'].astype(str) + "_" + data['gender'].astype(str)

N_REPEATS      = len(repeat_fold_cols)
N_OUTER_SPLITS = data[repeat_fold_cols[0]].nunique()
N_INNER_SPLITS = 5
base_random_state = 42

### 4. Helper functions

In [ ]:
# ==========================================
# 4. HELPERS
# ==========================================
def aggregate_mean_by_group(
    y: np.ndarray, p: np.ndarray, g: np.ndarray
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    y, p, g = np.asarray(y).astype(int), np.asarray(p).astype(float), np.asarray(g)
    uniq = np.unique(g)
    y_g  = np.zeros(len(uniq), dtype=int)
    p_g  = np.zeros(len(uniq), dtype=float)
    for i, gg in enumerate(uniq):
        idx    = np.where(g == gg)[0]
        p_g[i] = float(np.mean(p[idx])) if len(idx) else float('nan')
        y_g[i] = int(np.mean(y[idx]) >= 0.5) if len(idx) else 0
    return y_g, p_g, uniq

metrics_keys = ["accuracy", "f1", "auc", "precision", "recall", "brier"]

### 5. Nested cross-validation — setup

In [ ]:
# ==========================================
# 5. NESTED CV LOOP (ALL FEATURES)
# ==========================================
feature_cols = ALL_FEATURES

print(f"\n{'#'*80}")
print(f"  MODEL: SVM — ALL FEATURES  ({len(feature_cols)} features)")
print(f"  {feature_cols}")
print(f"{'#'*80}")

X_full          = data[feature_cols].copy()
y_full          = data['label']
y_stratify_full = data['stratify_key']

print(f"Feature matrix : {X_full.shape}")
print(f"Unique speakers: {len(np.unique(groups))}")
print(f"Repeats x Folds: {N_REPEATS} x {N_OUTER_SPLITS}")

results_sample  = {k: [] for k in metrics_keys}
results_speaker = {k: [] for k in metrics_keys}
conf_matrices_sample  = []
conf_matrices_speaker = []

# Per-fold (fold-to-fold) records, one row per (repeat, outer fold)
fold_records = []

### 5. Nested cross-validation — repeats × outer folds

In [ ]:
for repeat_idx, repeat_col in enumerate(repeat_fold_cols):
    repeat       = repeat_idx + 1
    current_seed = base_random_state + repeat_idx

    print(f"\n{'='*80}")
    print(f"[ALL_FEATURES] REPEAT {repeat}/{N_REPEATS}  "
          f"(Seed: {current_seed} | Column: {repeat_col})")
    print("="*80)

    for fold_id in range(N_OUTER_SPLITS):

        test_mask  = (data[repeat_col] == fold_id).values
        train_mask = ~test_mask

        X_train = X_full.iloc[train_mask]
        X_test  = X_full.iloc[test_mask]
        y_train = y_full.iloc[train_mask]
        y_test  = y_full.iloc[test_mask]
        g_train_np    = groups[train_mask]
        g_test_np     = groups[test_mask]
        y_strat_train = y_stratify_full.iloc[train_mask]

        # LEAKAGE CHECKS
        assert len(set(g_train_np) & set(g_test_np)) == 0, \
            f"❌ OUTER LEAKAGE R{repeat} Fold {fold_id}"
        assert len(set(g_train_np)) + len(set(g_test_np)) == len(np.unique(groups)), \
            f"❌ Speaker count mismatch R{repeat} Fold {fold_id}"
        assert len(set(np.where(train_mask)[0]) & set(np.where(test_mask)[0])) == 0, \
            f"❌ SAMPLE OVERLAP R{repeat} Fold {fold_id}"

        inner_cv_check = StratifiedGroupKFold(
            n_splits=N_INNER_SPLITS, shuffle=True, random_state=current_seed + fold_id,
        ).split(X_train, y_strat_train, g_train_np)
        for i, (tr_i, va_i) in enumerate(inner_cv_check):
            assert len(set(g_train_np[tr_i]) & set(g_train_np[va_i])) == 0, \
                f"❌ INNER LEAKAGE R{repeat} Fold {fold_id} Inner {i}"

        n_spk_train = len(set(g_train_np))
        n_spk_test  = len(set(g_test_np))
        print(f"R{repeat} Fold {fold_id+1:2d}: ✅ Checks passed | "
              f"Train: {len(X_train)} samples ({n_spk_train} spk) | "
              f"Test: {len(X_test)} samples ({n_spk_test} spk)")

        # GridSearchCV 
        svm_pipe = Pipeline([
            ('scaler', StandardScaler()),
            ('svc', SVC(probability=True, random_state=current_seed, class_weight='balanced'))
        ])
        param_grid = {
            "svc__C":      [0.1, 1, 10, 100, 1000],
            "svc__gamma":  ['scale', 1, 0.1, 0.01, 0.001, 0.0001],
            "svc__kernel": ['rbf'],
        }
        inner_cv = StratifiedGroupKFold(
            n_splits=N_INNER_SPLITS, shuffle=True, random_state=current_seed + fold_id,
        ).split(X_train, y_strat_train, g_train_np)

        grid_search = GridSearchCV(
            estimator=svm_pipe, param_grid=param_grid, cv=inner_cv,
            scoring="roc_auc", n_jobs=-1, verbose=0,
        )
        grid_search.fit(X_train, y_train)
        best_model  = grid_search.best_estimator_
        best_params = best_model.named_steps['svc'].get_params()

        # Sample-level predictions 
        y_pred_sample  = best_model.predict(X_test)
        y_proba_sample = best_model.predict_proba(X_test)[:, 1]

        results_sample["accuracy"].append(  accuracy_score(  y_test, y_pred_sample))
        results_sample["f1"].append(        f1_score(        y_test, y_pred_sample, zero_division=0))
        results_sample["auc"].append(       roc_auc_score(   y_test, y_proba_sample))
        results_sample["precision"].append( precision_score( y_test, y_pred_sample, zero_division=0))
        results_sample["recall"].append(    recall_score(    y_test, y_pred_sample, zero_division=0))
        results_sample["brier"].append(     brier_score_loss(y_test, y_proba_sample))
        conf_matrices_sample.append(confusion_matrix(y_test, y_pred_sample))

        # Speaker-level predictions 
        y_test_spk, y_proba_spk, _ = aggregate_mean_by_group(
            y_test.values, y_proba_sample, g_test_np
        )
        y_pred_spk = (y_proba_spk >= 0.5).astype(int)

        results_speaker["accuracy"].append(  accuracy_score(  y_test_spk, y_pred_spk))
        results_speaker["f1"].append(        f1_score(        y_test_spk, y_pred_spk, zero_division=0))
        results_speaker["auc"].append(       roc_auc_score(   y_test_spk, y_proba_spk))
        results_speaker["precision"].append( precision_score( y_test_spk, y_pred_spk, zero_division=0))
        results_speaker["recall"].append(    recall_score(    y_test_spk, y_pred_spk, zero_division=0))
        results_speaker["brier"].append(     brier_score_loss(y_test_spk, y_proba_spk))
        conf_matrices_speaker.append(confusion_matrix(y_test_spk, y_pred_spk))

        fold_records.append({
            "repeat":           repeat,
            "fold":             fold_id + 1,
            "seed":             current_seed,
            "best_C":           best_params['C'],
            "best_gamma":       best_params['gamma'],
            "best_kernel":      best_params['kernel'],
            "n_train_samples":  len(X_train),
            "n_test_samples":   len(X_test),
            "n_train_speakers": n_spk_train,
            "n_test_speakers":  n_spk_test,
            "sample_accuracy":  results_sample["accuracy"][-1],
            "sample_f1":        results_sample["f1"][-1],
            "sample_auc":       results_sample["auc"][-1],
            "sample_precision": results_sample["precision"][-1],
            "sample_recall":    results_sample["recall"][-1],
            "sample_brier":     results_sample["brier"][-1],
            "speaker_accuracy":  results_speaker["accuracy"][-1],
            "speaker_f1":        results_speaker["f1"][-1],
            "speaker_auc":       results_speaker["auc"][-1],
            "speaker_precision": results_speaker["precision"][-1],
            "speaker_recall":    results_speaker["recall"][-1],
            "speaker_brier":     results_speaker["brier"][-1],
        })

        print(f"           C={best_params['C']}, Gamma={best_params['gamma']}, "
              f"Kernel={best_params['kernel']}")
        print(f"           [Sample]  Acc: {results_sample['accuracy'][-1]:.4f} | "
              f"F1: {results_sample['f1'][-1]:.4f} | AUC: {results_sample['auc'][-1]:.4f}")
        print(f"           [Speaker] Acc: {results_speaker['accuracy'][-1]:.4f} | "
              f"F1: {results_speaker['f1'][-1]:.4f} | AUC: {results_speaker['auc'][-1]:.4f}")
        print("-" * 60)

### 6. Save per-fold results

In [ ]:
# ==========================================
# 6. SAVE FOLD-TO-FOLD RESULTS
# ==========================================
fold_df = pd.DataFrame(fold_records)

fold_csv = f"fold_results_svm_vowels.csv"
fold_df.to_csv(fold_csv, index=False)
print(f"\nSaved per-fold results → {fold_csv}")

fold_txt = f"fold_results_svm_vowels.txt"
with open(fold_txt, "w") as f:
    f.write(f"SVM — ALL FEATURES ({len(feature_cols)} features) — PC-GITA vowels\n")
    f.write(f"Features: {feature_cols}\n")
    f.write(f"Repeats x Folds: {N_REPEATS} x {N_OUTER_SPLITS}\n")
    f.write("=" * 100 + "\n")
    f.write(fold_df.to_string(index=False))
    f.write("\n")
print(f"Saved per-fold results → {fold_txt}")

### 7. Summary over all folds

In [ ]:
# ==========================================
# 7. SUMMARY TABLE (aggregated over all folds)
# ==========================================
total_folds = N_REPEATS * N_OUTER_SPLITS

print(f"\n\n{'='*80}")
print(f"SVM — ALL FEATURES — FINAL RESULTS ({total_folds} Total Folds)")
print("="*80)

summary_rows = []
for level, r in (("sample", results_sample), ("speaker", results_speaker)):
    row = {"level": level}
    for metric in metrics_keys:
        row[f"{metric}_mean"] = np.mean(r[metric])
        row[f"{metric}_std"]  = np.std(r[metric])
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)

for level in ("sample", "speaker"):
    print(f"\n--- {level.upper()}-LEVEL ---")
    sub = summary_df[summary_df["level"] == level].set_index("level")
    for m in metrics_keys:
        mu  = sub.loc[level, f"{m}_mean"]
        std = sub.loc[level, f"{m}_std"]
        print(f"  {m.capitalize():<12}: {mu:.4f} ± {std:.4f}")

summary_csv = f"results_svm_vowels.csv"
summary_df.to_csv(summary_csv, index=False)
print(f"\nSaved aggregated summary → {summary_csv}")

### Aggregated confusion matrices

In [ ]:
# Confusion matrices 
print(f"\n{'='*80}")
print("AGGREGATED CONFUSION MATRICES")
print("="*80)
cm_s = np.sum(conf_matrices_sample,  axis=0)
cm_k = np.sum(conf_matrices_speaker, axis=0)
print(f"\nSample-Level:\n{cm_s}")
print(f"\nSpeaker-Level:\n{cm_k}")